# NIH Chest X-ray GPU fine-tune

Runtime → Change runtime type → **T4 GPU**, then Run all.
Downloads the NIH images from the Hugging Face mirror, builds the manifest, and fine-tunes ResNet-50 for 10 epochs
with class-balanced focal loss and rare-class oversampling.
Each epoch logs macro-F1, balanced accuracy, and Nodule/Pneumonia/Edema/Fibrosis recall.
The next backbone experiment is configs/multiclass.densenet121.yaml, after this run finishes.

In [ ]:
import torch
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit('GPU runtime is required. Runtime -> Change runtime type -> T4 GPU')

In [ ]:
import os, subprocess
from pathlib import Path
root = Path('/content/ai-ing')
if not (root / 'train_multiclass.py').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', 'cursor/cb-focal-rare-eval-4f0c', 'https://github.com/aninsung/ai-ing.git', str(root)])
os.chdir(root)
subprocess.check_call(['pip', 'install', '-q', 'pyyaml', 'scikit-learn'])
print('repo', root)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, 'scripts/download_nih_data.py', '--root', 'data'])

In [ ]:
from pathlib import Path
data = Path('data')
archive = data / 'archive'
if archive.is_symlink() or archive.exists():
    if archive.is_symlink() or archive.is_file():
        archive.unlink()
    else:
        import shutil
        shutil.rmtree(archive)
img_dir = archive / 'images-224' / 'images-224'
img_dir.mkdir(parents=True)
for src, name in (
    (data / 'Data_Entry_2017.csv', 'Data_Entry_2017.csv'),
    (data / 'train_val_list.txt', 'train_val_list_NIH.txt'),
    (data / 'test_list.txt', 'test_list_NIH.txt'),
):
    (archive / name).symlink_to(src.resolve())
linked = 0
for path in (data / 'raw').rglob('*.png'):
    dest = img_dir / path.name
    if not dest.exists():
        dest.symlink_to(path.resolve())
    linked += 1
print('pngs', linked)
import subprocess, sys
subprocess.check_call([sys.executable, 'prepare_multiclass_manifest.py', '--archive', 'data/archive', '--output', 'data/nih_multiclass_manifest.csv'])

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-u', 'train_multiclass.py', '--config', 'configs/multiclass.cb_focal.yaml'])